# Writing simpleNMR assignments back to JASON: what works

**Ethylbenzene test, October 2026 — beautifuljason 1.3.0b1**

We assigned ethylbenzene's experimental shifts in a JASON document by hand, one step at a time,
to find out which assignments JASON accepts and what makes it draw the HMBC and COSY
correlations on the molecule. This notebook records the results and reproduces the final,
working result in one run.

## Summary

A complete, correct display needs three things:

1. **Carbons → their 1D 13C peak.**
2. **Protons → their HSQC peak**, on the 1H axis (`dim=0`).
3. **Every HMBC and COSY peak moved onto the positions of the signals its two ends are assigned to.**
   JASON regroups the signals when it loads the file and then draws each correlation itself.

HMBC and COSY peaks are **not** assigned; JASON derives the correlation lines from the assigned
1D and HSQC signals.

## The document

Atom numbers are 1-based, as in simpleNMR and the molfile (beautifuljason counts from 0):
1, 3 ortho · 2 quaternary · 4, 6 meta · 5 para · 7 CH₂ · 8 CH₃.

| `nmr_data` | experiment | used for |
|---|---|---|
| [0] | 1H 1D | — (no picked peaks) |
| [1] | 13C 1D | carbon assignments |
| [2] | COSY | correlations (moved) |
| [3] | 1H pure shift | — (cannot be assigned, see below) |
| [4] | HMBC | correlations (moved) |
| [5] | 1H/13C 2D, empty | — |
| [6] | HSQC | proton assignments |

JASON's predictions have **one shift per atom**, including each atom of a symmetric pair
(C1 and C3 are separate shifts) and one unmarked shift for the CH₂ protons.

## How JASON handles assignments

* `molecule.set_assignments([(shift, signal), (shift, peak, dim), ...])` replaces the assignments.
* Behind every assignable signal is a **merged signal**: JASON's grouping of peaks from different
  spectra that sit at the same position (a carbon's 1D peak, its HSQC peak, HMBC peaks on its 13C axis...).
  `set_assignments` looks the signal up in these groups.
* **Correlation lines come from merged 2D signals**: JASON draws an HMBC or COSY line once the merged
  signals at both ends carry an assigned shift.
* JASON builds the groups from peak *positions*. The same proton sits at slightly different positions
  in different experiments (meta H: 7.4012 in the HSQC, 7.4039 in the HMBC, 7.4051 in the COSY),
  and the HMBC 13C axis is coarse, so some correlation peaks land in the wrong group or in none.
  That is what moving the peaks fixes.

A merged signal's ID can be the same as the ID of a peak in it: C8's 1D 13C peak and its merged signal
share `{84fce28e-…}`.

## Results, step by step

Each step started from a fresh copy of the original document.

| step | what was done | result in JASON |
|---|---|---|
| 1 | C8, C2 → 1D 13C peaks; H8 → HSQC peak | ✓ values shown; HMBC CH₃ → C2 appears **without assigning any HMBC peak** |
| 3 | + C7 → 1D 13C peak; H7 → HSQC peak | ✓ HMBC CH₃ → C7 and COSY CH₂–CH₃ appear; HMBC CH₂ → C2 missing |
| 4 | + aromatic carbons → 1D 13C; aromatic H → HSQC (both atoms of each symmetric pair on the same peaks) | ✓ all values shown; COSY meta–para appears; meta–ortho COSY, CH₂ → C2, CH₂ → ortho, meta-H → C2 missing; para-H → ortho drawn as para-H → **meta** |
| 5 | + HMBC meta-H → C2 peak moved onto (H meta HSQC, C2 1D) | ✓ meta-H → C2 appears, to both meta carbons |
| 6 | + every other HMBC and COSY peak moved likewise | ✓ **all correlations correct** |

### What does not work

| attempt | JASON's message |
|---|---|
| proton → pure-shift peak | "Cannot find a matching merged 1D signal" (pure-shift peaks are in no merged signal in this document) |
| proton → HSQC multiplet (2D integral) | "Only NMRPeak objects can be assigned from a 2D spectrum" |
| one shift → its HSQC peak *and* HMBC/COSY peaks in other groups (earlier writer run) | "A chemical shift cannot be assigned to more than one signal" |

Not tested: assigning an HMBC/COSY peak explicitly as well as the 1D/HSQC signal (it is not needed
for the lines to appear), and protons → 1H multiplets (the 1H 1D had none picked).

## Reproduce the final result

The cells below apply the whole recipe to a fresh copy of the original document in one go and save
`ethylbenzene_recipe.jjh5`. They also write `ethylbenzene_recipe_moves.json`, a record of every peak's
original and new position, so the edit to the peak table is documented and reversible.

In [1]:
from pathlib import Path
import json
import shutil
import beautifuljason as bjason

# The document as JASON exported it (the backup from a run, never a live input.jjh5)
SOURCE = Path("/Users/vsmw51/Documents/JASON/External Tools/simpleNMRverify/"
              "2026-10-05T18-57-21.380_60447/input.20261005-185758.bak.jjh5")
OUT_DIR = Path.home() / "Documents" / "ethylbenzene_tests"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "ethylbenzene_recipe.jjh5"

H1 = bjason.Molecule.Atom.NuclType.H1
C13 = bjason.Molecule.Atom.NuclType.C13
print("beautifuljason", bjason.__version__)

beautifuljason 1.3.0b1


### The data

Carbon and proton assignments: one row per atom, with the JEOL ID of the peak it is assigned to.
Symmetric partners share a peak. Protons use the HSQC peak's 1H axis (`dim=0`).

In [2]:
#            atom  JEOL ID of the 13C 1D peak                  JEOL ID of the HSQC peak
ATOMS = [
    dict(atom=1, label="ortho", c13="{981e8a05-cc8d-479a-a660-57d1de40ed90}", hsqc="{b02fad61-a90c-43ae-9115-ebbc664d9306}"),
    dict(atom=2, label="quat",  c13="{db5d5e8d-a444-4c9f-8652-052226a52d1a}", hsqc=None),
    dict(atom=3, label="ortho", c13="{981e8a05-cc8d-479a-a660-57d1de40ed90}", hsqc="{b02fad61-a90c-43ae-9115-ebbc664d9306}"),
    dict(atom=4, label="meta",  c13="{e0ce5f38-c916-452d-ac68-55a58caf9cc6}", hsqc="{fc3bd16a-e67c-47d6-bc1d-73daf2d26bc1}"),
    dict(atom=5, label="para",  c13="{7dd23078-a81f-4aca-a9e8-ceef2c501e24}", hsqc="{f80dacb4-d950-47eb-8ad9-7a8b46596427}"),
    dict(atom=6, label="meta",  c13="{e0ce5f38-c916-452d-ac68-55a58caf9cc6}", hsqc="{fc3bd16a-e67c-47d6-bc1d-73daf2d26bc1}"),
    dict(atom=7, label="CH2",   c13="{60d8daa1-0725-4220-9bc5-0ed42dc2e803}", hsqc="{bfb0db5f-1680-4585-aa52-955107815816}"),
    dict(atom=8, label="CH3",   c13="{84fce28e-910a-4c13-bf36-b2839f6ad45a}", hsqc="{49dad087-f74e-45da-ada3-1bd45abb1bbc}"),
]

Correlation peaks to move. `new` gives the target on each axis: the position of the HSQC peak for a
proton end, of the 1D 13C peak for a carbon end. Axis 0 is 1H in both the HMBC and the COSY here.

In [3]:
MOVES = [
    #                                 JEOL ID                                     new position  {axis: ppm}
    dict(label="HMBC meta-H -> C2",    jeol_id="{1774e688-a307-4f0c-92ad-b60b6da1c87b}", new={0: 7.4012, 1: 144.3054}),
    dict(label="HMBC CH3 -> C2",       jeol_id="{bb84d936-9adf-44bd-8c46-ebd0a7aad069}", new={0: 1.3676, 1: 144.3054}),
    dict(label="HMBC CH3 -> C7",       jeol_id="{17aa3cdf-e8f8-478b-b07c-124e637c561c}", new={0: 1.3676, 1: 28.9897}),
    dict(label="HMBC CH2 -> C2",       jeol_id="{9cb4a1c8-fb58-4864-a418-a446a0803e6e}", new={0: 2.7775, 1: 144.3054}),
    dict(label="HMBC CH2 -> ortho",    jeol_id="{82292e56-7132-4af1-a2df-269c3383b84b}", new={0: 2.7775, 1: 127.9292}),
    dict(label="HMBC para-H -> ortho", jeol_id="{00a416e8-712d-48b2-b645-73536f816579}", new={0: 7.2956, 1: 127.9292}),
    dict(label="COSY CH2 - CH3",       jeol_id="{c722cbc6-e190-4a44-b790-757bcb2691ab}", new={0: 2.7775, 1: 1.3676}),
    dict(label="COSY meta - ortho",    jeol_id="{0ce9e917-3c5f-4d11-8297-93b930ab33fd}", new={0: 7.4012, 1: 7.3226}),
    dict(label="COSY meta - para",     jeol_id="{e48c637d-1420-4c76-941c-39c6c30dd5a8}", new={0: 7.4012, 1: 7.2956}),
]

### Helpers

In [4]:
def find_peak(doc, jeol_id):
    """The peak with this JEOL ID, searching every spectrum."""
    for spectrum in doc.nmr_data:
        try:
            for peak in spectrum.peaks:
                if str(peak.id) == jeol_id:
                    return peak
        except KeyError:            # spectrum without a peak table
            pass
    raise LookupError(f"no peak with ID {jeol_id}")


def find_shift(mol, nucleus, atom):
    """JASON's predicted shift for this nucleus on atom (1-based). One shift per atom in this document."""
    for spectrum in mol.spectra:
        if spectrum.nucleus == nucleus:
            for shift in spectrum.shifts:
                if atom - 1 in [int(n) for n in shift.nums]:
                    return shift
    raise LookupError(f"no predicted {nucleus.name} shift on atom {atom}")


def fmt(values):
    return "(" + ", ".join(f"{float(v):.4f}" for v in values) + ")"

### Apply the recipe

In [5]:
shutil.copy2(SOURCE, OUT)
log = []

with bjason.Document(str(OUT), mode="r+") as doc:
    mol = doc.mol_data[0]

    # 1 + 2: carbons -> 1D 13C peak, protons -> HSQC peak on the 1H axis
    assignments = []
    for a in ATOMS:
        assignments.append((find_shift(mol, C13, a["atom"]), find_peak(doc, a["c13"])))
        if a["hsqc"]:
            assignments.append((find_shift(mol, H1, a["atom"]), find_peak(doc, a["hsqc"]), 0))

    # 3: move each correlation peak onto the positions of its two assigned ends
    for m in MOVES:
        peak = find_peak(doc, m["jeol_id"])
        old = peak.h5_group.attrs["Pos"].copy()
        new = old.copy()
        for axis, value in m["new"].items():
            new[axis] = value
        peak.h5_group.attrs["Pos"] = new
        log.append(dict(label=m["label"], jeol_id=m["jeol_id"],
                        original=[float(x) for x in old], moved_to=[float(x) for x in new]))
        print(f"moved {m['label']:22} {fmt(old)} -> {fmt(new)}")

    mol.set_assignments(assignments)
    print(f"\n{len(assignments)} assignments written")

(OUT_DIR / "ethylbenzene_recipe_moves.json").write_text(json.dumps(log, indent=2))
print(f"Saved {OUT} - open it in JASON")

moved HMBC meta-H -> C2      (7.4039, 144.4006, 0.0000) -> (7.4012, 144.3054, 0.0000)
moved HMBC CH3 -> C2         (1.3680, 144.4002, 0.0000) -> (1.3676, 144.3054, 0.0000)
moved HMBC CH3 -> C7         (1.3681, 28.9667, 0.0000) -> (1.3676, 28.9897, 0.0000)
moved HMBC CH2 -> C2         (2.7748, 144.3940, 0.0000) -> (2.7775, 144.3054, 0.0000)
moved HMBC CH2 -> ortho      (2.7752, 127.9209, 0.0000) -> (2.7775, 127.9292, 0.0000)
moved HMBC para-H -> ortho   (7.2955, 127.9546, 0.0000) -> (7.2956, 127.9292, 0.0000)
moved COSY CH2 - CH3         (2.7759, 1.3684, 0.0000) -> (2.7775, 1.3676, 0.0000)
moved COSY meta - ortho      (7.4051, 7.3263, 0.0000) -> (7.4012, 7.3226, 0.0000)
moved COSY meta - para       (7.4047, 7.2966, 0.0000) -> (7.4012, 7.2956, 0.0000)

15 assignments written
Saved /Users/vsmw51/Documents/ethylbenzene_tests/ethylbenzene_recipe.jjh5 - open it in JASON


## What this means for simpleNMR → JASON

* **The writer needs no ppm matching** if simpleNMR carries JEOL IDs through:
  * `json_converter.py` adds each peak's `ID` attribute to the input JSON;
  * simpleNMR keeps it in its experiment dataframes and returns, per experimental group, the IDs of its
    1D 13C peak and HSQC peak, plus every HMBC/COSY peak it used with its snapped positions
    (`f1_ppm`/`f2_ppm`; the originals are in `*_orig`);
  * the export keys this by the node `id`, which travels with a shift when the user moves it in the viewer.
* **simpleNMR already computes the aligned positions**: its `snap()` step moves every HMBC/COSY peak to
  the standard carbon and proton values. When there is a 1D 13C those are the 1D values, matching what we
  assign. Without one, the carbon target is the position of the signal the carbon is assigned to
  (HSQC 13C axis, or the chosen HMBC peak for a quaternary carbon).
* The current writer's rule (carbons → 1D 13C, else HSQC, else HMBC; protons → HSQC) stays; the change is
  finding peaks by ID and adding the peak moves.

## Caution

Moving peaks edits the experimental peak table to fit the assignment. The shifts are small (here at most
0.095 ppm in 13C and 0.004 ppm in 1H,) and `ShapePar` was `[0, 0, 0]` for every
moved peak, so no position is duplicated elsewhere in the peak. A record of the originals (the JSON above,
or the peak's `PeakLabel`) keeps the change visible and reversible.

## Questions for JEOL

1. Is there a merge tolerance (in JASON or the API) that would group these peaks without moving them?
2. Why was the HMBC peak at 127.95 ppm grouped with the meta carbon (128.38) rather than ortho (127.93)?
3. Is moving peak positions in the file, and letting JASON regroup on load, a supported way to do this?
4. Can pure-shift peaks, or 1H multiplets, be made part of merged signals so protons can be assigned to them?
5. Does `set_assignments` replace all existing assignments, or add to them?